<div align="center">

  <a href="https://grayboxtech.github.io/weightslab/latest/index.html" target="_blank">
    <img width="100%" src="https://raw.githubusercontent.com/GrayboxTech/.github/main/profile/weightslab-banner-light.png" alt="WeightsLab banner"></a>

  <a href="https://github.com/GrayboxTech/weightslab/blob/main/LICENSE"><img src="https://img.shields.io/badge/License-Apache%202.0-blue.svg" alt="License"></a>
  <a href="https://github.com/GrayboxTech/weightslab/stargazers"><img src="https://img.shields.io/github/stars/GrayboxTech/weightslab?style=flat&color=5865F2" alt="Stars"></a>
  <a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/v/weightslab?style=flat&color=5865F2&logo=pypi&logoColor=white" alt="Version"></a>
  <br>
  <a href="https://colab.research.google.com/github/GrayboxTech/weightslab/blob/main/weightslab/examples/Notebooks/PyTorch/ws-generation.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open WeightsLab In Colab"></a>

  Welcome to the WeightsLab <b>Image Generation (VAD U-Net)</b> notebook! <a href="https://github.com/GrayboxTech/weightslab">WeightsLab</a> traces training signals back to the exact samples producing them. Browse the <a href="https://grayboxtech.github.io/weightslab/latest/index.html">Docs</a> for details.</div>

# Image Generation (VAD U-Net) with WeightsLab

Trains a U-Net-style generator and tracks per-sample reconstruction signals.

> Data: **This example needs your own image dataset** - set `data_root` in the config to a folder of images before running (there is no bundled/downloaded dataset for it).

Unlike the self-contained classification demo, this example uses local helper modules
(`utils/`) and bundled/downloaded data, so the notebook **clones the repo** and runs the
example in place.

## Setup

On Colab, pick a **T4 GPU** runtime (`Runtime -> Change runtime type -> T4 GPU`).

<a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/v/weightslab?color=5865F2&logo=pypi&logoColor=white" alt="PyPI - Version"></a>
<a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/dm/weightslab?color=5865F2" alt="PyPI - Downloads"></a>
<a href="https://pypi.org/project/weightslab/"><img src="https://img.shields.io/pypi/pyversions/weightslab?color=5865F2&logo=python&logoColor=white" alt="PyPI - Python Version"></a>

In [ ]:
# Clone the repo (for the example's utils/ + data) and install WeightsLab.
import os
if not os.path.isdir("weightslab"):
    !git clone --branch torch_collab_examples https://github.com/GrayboxTech/weightslab.git
%pip install -q ./weightslab
!pip install --upgrade "protobuf>=6.31.1"
%pip install "torchvision>=0.16"

In [ ]:
# Move into the example directory so its `utils/` and config.yaml resolve.
%cd weightslab/weightslab/examples/PyTorch/ws-generation
# Install the example's own extra requirements, if any.
import os
if os.path.exists("requirements.txt"):
    !pip install -q -r requirements.txt

## Configuration

Every tunable lives in **`config.yaml`** (already commented). We load it into a `config` dict
so you can tweak values here; the changes are written back before training.

In [ ]:
import yaml

with open("config.yaml") as f:
    config = yaml.safe_load(f)

# --- tweak any parameter here, e.g. ---
# config["eval_full_to_train_steps_ratio"] = 5
config["serving_grpc"] = True   # expose the gRPC backend for Weights Studio

with open("config.yaml", "w") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print(yaml.safe_dump(config, sort_keys=False))

## (Optional) Expose the backend for the live UI

To watch training **live in Weights Studio on your own machine**, run this cell first (it opens a
raw-TCP [`bore`](https://github.com/ekzhang/bore) tunnel over the free public relay `bore.pub` -
**no signup, no card**), then start training below.

> `bore.pub` is a shared public relay; the random port is the only thing protecting your endpoint.
> Fine for a demo, not for sensitive data.

In [ ]:
import os, re, tarfile, threading, urllib.request, subprocess

EXPOSE_UI = True          # set False for a headless run (no tunnel)
BACKEND_PORT = 50051      # gRPC port to forward

endpoint = None
if EXPOSE_UI:
    bore = os.path.join(os.getcwd(), "bore")
    if not os.path.exists(bore):
        BORE = "v0.6.0"
        urllib.request.urlretrieve(
            f"https://github.com/ekzhang/bore/releases/download/{BORE}/bore-{BORE}-x86_64-unknown-linux-musl.tar.gz",
            "bore.tar.gz")
        with tarfile.open("bore.tar.gz") as t:
            t.extractall()
        os.chmod(bore, 0o755)

    proc = subprocess.Popen(
        [bore, "local", str(BACKEND_PORT), "--to", "bore.pub"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        m = re.search(r"bore\.pub:(\d+)", line)
        if m:
            endpoint = f"bore.pub:{m.group(1)}"
            break
    threading.Thread(target=lambda: [None for _ in proc.stdout], daemon=True).start()

    print("=" * 60)
    print("Backend exposed at:", endpoint)
    print("On your OWN machine (Docker running), run:")
    print("    weightslab ui launch")
    print(f"    weightslab tunnel {endpoint}")
    print("=" * 60)
else:
    print("EXPOSE_UI is False - no tunnel.")

## Train

This runs the example's `main.py`, which wraps the model / optimizer / loaders / losses with
`wl.watch_or_edit(...)`, starts the gRPC backend, and trains while streaming per-sample signals.

> Training runs for `training_steps` (default 200000); interrupt the cell to stop early. Interrupt the cell (stop button) to stop training; the exported history and
> dataframe are written to a temp `root_log_dir` (printed at startup).

In [ ]:
!python main.py

## See it live in Weights Studio

**Colab has no Docker daemon**, so run Studio on your own machine and point it at this notebook's
backend using the `bore.pub:<port>` printed in the Expose section:

```bash
pip install weightslab
weightslab ui launch                       # Terminal 1 - opens http://localhost:5173
weightslab tunnel bore.pub:12345           # Terminal 2 - the host:port from the Expose section
```

Then open **http://localhost:5173**. Prefer local-only? Run this example directly on your machine
(`weightslab start example` selects a bundled example) and launch the UI next to it - no tunnel.

---

<div align="center">
Crafted by <a href="https://github.com/GrayboxTech/weightslab">GrayboxTech</a> - if WeightsLab helps you catch a bad label, drop us a star on <a href="https://github.com/GrayboxTech/weightslab">GitHub</a>.
</div>